# Saving forecast results to DuckDB

In [1]:
import pandas as pd
import duckdb

# Load forecast CSV (path from notebooks/ → data/processed)
df_fc = pd.read_csv("../data/processed/texas_forecast_12m.csv", parse_dates=["Date"])
df_fc.head()


,Date,state,value,source,horizon
0,2025-07-01,Texas,443785.06,forecast,1
1,2025-08-01,Texas,451922.75,forecast,2
2,2025-09-01,Texas,420523.25,forecast,3
3,2025-10-01,Texas,418446.75,forecast,4
4,2025-11-01,Texas,387949.66,forecast,5


In [ ]:
#connect to DuckDB and create the gas_forecast
con = duckdb.connect("../natural_gas.duckdb")

con.execute("""
CREATE TABLE IF NOT EXISTS gas_forecast (
    Date DATE,
    state VARCHAR,
    value DOUBLE,
    source VARCHAR,
    horizon INTEGER
);
""")


In [ ]:
# Delete existing Texas forecast data (so we don't duplicate on re-runs)
con.execute("""
DELETE FROM gas_forecast
WHERE state = 'Texas' AND source = 'forecast';
""")


In [ ]:
# Insert new forecast data
con.execute("""
INSERT INTO gas_forecast
SELECT Date, state, value, source, horizon
FROM df_fc;
""")


In [ ]:
# Query to verify insertion
con.execute("""
SELECT *
FROM gas_forecast
WHERE state = 'Texas'
ORDER BY Date
""").df()


,Date,state,value,source,horizon
0,2025-07-01,Texas,443785.06,forecast,1
1,2025-08-01,Texas,451922.75,forecast,2
2,2025-09-01,Texas,420523.25,forecast,3
3,2025-10-01,Texas,418446.75,forecast,4
4,2025-11-01,Texas,387949.66,forecast,5
5,2025-12-01,Texas,429292.80,forecast,6
6,2026-01-01,Texas,457419.06,forecast,7
7,2026-02-01,Texas,414897.47,forecast,8
8,2026-03-01,Texas,383811.22,forecast,9
9,2026-04-01,Texas,376013.72,forecast,10


In [ ]:
con.close()